# NB 13 — Sparse PCA per-block + Régression logistique multinomiale (R)

## Objet

Tester le pipeline minimaliste : **Sparse PCA per-block** comme réducteur de dimension supervisé, puis **régression logistique multinomiale avec elastic net** comme classifieur final. Pas de cooperative learning.

## Motivation : isoler l'apport de cooperative

Cooperative learning fait deux choses simultanément :
1. **Sélection de variables sparse** (via L1 sur les coefficients)
2. **Agrément cross-blocs** (via le terme $\frac{\rho}{2}\|X\theta_x - Z\theta_z\|^2$)

Mais Sparse PCA fait aussi de la sélection de variables sparse (chargements creux). Si on combine Sparse PCA + cooperative comme dans NB12, **on sélectionne deux fois** : une fois non-supervisée (Sparse PCA), puis une fois supervisée (cooperative L1). C'est peut-être redondant.

NB13 teste la version **simple** : Sparse PCA per-block (sélection sparse) suivi de **régression logistique multinomiale** (modèle prédictif, pas de sélection supplémentaire). Si NB13 ≈ NB12, alors cooperative n'apporte rien au-delà de la double-sélection.

## Différences-clé avec les notebooks précédents

| Notebook | Pré-traitement | Classifieur | Mode multi-classe |
|---|---|---|---|
| NB04 v2 | Sparse PCA sur **concat** | LogReg ElasticNet | Multinomial natif |
| NB10 | Cooperative OvR (raw) | LDA | OvR + LDA |
| NB11 | Cooperative OvR (raw) | argmax | OvR |
| NB12 | Sparse PCA per-block + Cooperative OvR | argmax | OvR |
| **NB13** | **Sparse PCA per-block** | **LogReg multinomiale** | **Multinomial natif (glmnet)** |

L'avantage de NB13 : `glmnet::glmnet` supporte le **multinomial directement** (contrairement à `multiview` qu'on a dû contourner via OvR). Pipeline plus court et conceptuellement plus propre.


## 1. Setup

In [1]:
set.seed(42)
SEED <- 42
LABEL_ORDER <- c("cort", "dipg", "midl")

cran_packages <- c("glmnet", "data.table", "caret", "elasticnet")
to_install <- cran_packages[!vapply(cran_packages, requireNamespace,
                                    logical(1), quietly = TRUE)]
if (length(to_install) > 0) {
  install.packages(to_install, repos = "https://cloud.r-project.org")
}

USE_MIXOMICS <- requireNamespace("mixOmics", quietly = TRUE)

suppressPackageStartupMessages({
  library(glmnet)
  library(data.table)
  library(caret)
  library(elasticnet)
  if (USE_MIXOMICS) library(mixOmics)
})

cat("glmnet     :", as.character(packageVersion("glmnet")), "\n")
cat("elasticnet :", as.character(packageVersion("elasticnet")), "\n")
cat("USE_MIXOMICS :", USE_MIXOMICS, "\n")


glmnet     : 5.0 
elasticnet : 1.3 
USE_MIXOMICS : TRUE 


## 2. Chargement des données

In [2]:
root <- normalizePath(file.path(getwd()), winslash = "/", mustWork = FALSE)
data_dir <- if (dir.exists(file.path(root, "data"))) file.path(root, "data") else file.path(dirname(root), "data")

to_numeric_frame <- function(df) {
  rn <- rownames(df)
  out <- as.data.frame(
    lapply(df, function(x) as.numeric(gsub(",", ".", as.character(x), fixed = TRUE))),
    check.names = FALSE
  )
  rownames(out) <- rn
  out
}

extract_id_column <- function(df) if ("row_id" %in% names(df)) "row_id" else names(df)[1]

load_block <- function(block_name, split) {
  path <- file.path(data_dir,
    sprintf("ge_cgh_locIGR__multiblocks__%s__%s.csv", block_name, split))
  df <- as.data.frame(data.table::fread(path, check.names = FALSE))
  id_col <- extract_id_column(df)
  rownames(df) <- as.character(df[[id_col]])
  df[[id_col]] <- NULL
  to_numeric_frame(df)
}

load_targets <- function(split) {
  path <- file.path(data_dir,
    sprintf("ge_cgh_locIGR__multiblocks__y__%s.csv", split))
  y_df <- as.data.frame(data.table::fread(path, check.names = FALSE))
  id_col <- extract_id_column(y_df)
  rownames(y_df) <- as.character(y_df[[id_col]])
  y_df[[id_col]] <- NULL
  targets <- factor(
    LABEL_ORDER[max.col(as.matrix(y_df[, LABEL_ORDER]), ties.method = "first")],
    levels = LABEL_ORDER
  )
  names(targets) <- rownames(y_df)
  targets
}

fill_missing_from_train <- function(train_df, test_df) {
  medians <- vapply(train_df, median, numeric(1), na.rm = TRUE)
  for (col in names(train_df)) {
    train_df[[col]][is.na(train_df[[col]])] <- medians[[col]]
    test_df[[col]][is.na(test_df[[col]])] <- medians[[col]]
  }
  list(train = train_df, test = test_df)
}

X_ge_train  <- load_block("GE",  "train")
X_ge_test   <- load_block("GE",  "test")
X_cgh_train <- load_block("CGH", "train")
X_cgh_test  <- load_block("CGH", "test")
y_train     <- load_targets("train")
y_test      <- load_targets("test")

train_ids <- Reduce(intersect, list(rownames(X_ge_train), rownames(X_cgh_train), names(y_train)))
test_ids  <- Reduce(intersect, list(rownames(X_ge_test),  rownames(X_cgh_test),  names(y_test)))

X_ge_train  <- as.matrix(X_ge_train [train_ids, , drop = FALSE])
X_cgh_train <- as.matrix(X_cgh_train[train_ids, , drop = FALSE])
y_train     <- y_train [train_ids]
X_ge_test   <- as.matrix(X_ge_test  [test_ids,  , drop = FALSE])
X_cgh_test  <- as.matrix(X_cgh_test [test_ids,  , drop = FALSE])
y_test      <- y_test   [test_ids]

# Imputation médiane train
filled_ge  <- fill_missing_from_train(as.data.frame(X_ge_train), as.data.frame(X_ge_test))
X_ge_train <- as.matrix(filled_ge$train);  X_ge_test <- as.matrix(filled_ge$test)
filled_cgh <- fill_missing_from_train(as.data.frame(X_cgh_train), as.data.frame(X_cgh_test))
X_cgh_train <- as.matrix(filled_cgh$train); X_cgh_test <- as.matrix(filled_cgh$test)

cat(sprintf("Train: %d patients | Test: %d patients\n", length(y_train), length(y_test)))
cat(sprintf("GE: %d features | CGH: %d features\n",
            ncol(X_ge_train), ncol(X_cgh_train)))


Train: 39 patients | Test: 14 patients
GE: 15702 features | CGH: 1229 features


## 3. Sparse PCA per-block (wrapper identique à NB12)

Centrage et scaling manuels pour robustesse, puis Sparse PCA via mixOmics (préférence) ou elasticnet (fallback CRAN).

In [3]:
fit_spca <- function(X, ncomp, keep_per_comp) {
  X_center <- colMeans(X)
  X_scale  <- apply(X, 2, sd)
  X_scale[X_scale < 1e-10] <- 1
  X_scaled <- sweep(sweep(X, 2, X_center, "-"), 2, X_scale, "/")

  if (USE_MIXOMICS) {
    fit <- mixOmics::spca(
      X_scaled, ncomp = ncomp,
      keepX = rep(keep_per_comp, ncomp),
      scale = FALSE, center = FALSE
    )
    loadings <- as.matrix(fit$loadings$X)
    var_expl <- tryCatch(as.numeric(fit$prop_expl_var$X), error = function(e) NULL)
    backend  <- "mixOmics"
  } else {
    fit <- elasticnet::spca(
      X_scaled, K = ncomp,
      para = rep(keep_per_comp, ncomp),
      type = "predictor", sparse = "varnum",
      lambda = 1e-6, use.corr = FALSE, trace = FALSE
    )
    loadings <- as.matrix(fit$loadings)
    var_expl <- NULL
    backend  <- "elasticnet"
  }

  variates <- X_scaled %*% loadings
  list(backend = backend, loadings = loadings, variates = variates,
       center = X_center, scale = X_scale, var_expl = var_expl)
}

project_spca <- function(X_new, spca_obj) {
  X_scaled <- sweep(sweep(X_new, 2, spca_obj$center, "-"), 2, spca_obj$scale, "/")
  X_scaled %*% spca_obj$loadings
}


## 4. Sparse PCA per-block sur les données train

In [4]:
N_COMP_GE  <- 10
N_COMP_CGH <- 5
KEEP_GE    <- 80
KEEP_CGH   <- 30

cat("\nSparse PCA bloc GE...\n")
spca_ge  <- fit_spca(X_ge_train,  N_COMP_GE,  KEEP_GE)

cat("Sparse PCA bloc CGH...\n")
spca_cgh <- fit_spca(X_cgh_train, N_COMP_CGH, KEEP_CGH)

# Composantes train et test (test projeté sur loadings train)
Z_ge_train  <- spca_ge$variates
Z_cgh_train <- spca_cgh$variates
Z_ge_test   <- project_spca(X_ge_test,  spca_ge)
Z_cgh_test  <- project_spca(X_cgh_test, spca_cgh)

colnames(Z_ge_train)  <- paste0("GE_PC",  seq_len(ncol(Z_ge_train)))
colnames(Z_cgh_train) <- paste0("CGH_PC", seq_len(ncol(Z_cgh_train)))
colnames(Z_ge_test)   <- colnames(Z_ge_train)
colnames(Z_cgh_test)  <- colnames(Z_cgh_train)

# Concaténation des composantes : matrice 39 × 15
Z_train <- cbind(Z_ge_train, Z_cgh_train)
Z_test  <- cbind(Z_ge_test,  Z_cgh_test)

cat(sprintf("\nDim Z_train (composantes concaténées) : %d × %d\n",
            nrow(Z_train), ncol(Z_train)))



Sparse PCA bloc GE...


Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”


Sparse PCA bloc CGH...


Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”



Dim Z_train (composantes concaténées) : 39 × 15


## 5. Sanity check — régression logistique multinomiale unique sur tout le train

`glmnet` supporte nativement `family = "multinomial"`. On utilise `cv.glmnet` pour choisir λ par CV interne, avec `alpha = 0.5` (Elastic Net 50/50 L1/L2).

`type.multinomial = "grouped"` force la sparsité au niveau **variable** : un coefficient est sélectionné simultanément pour les 3 classes (utile pour l'interprétation — on identifie des composantes globalement importantes plutôt que des coefficients par classe).

In [5]:
set.seed(SEED)
fit_test <- cv.glmnet(
  x      = Z_train,
  y      = y_train,
  family = "multinomial",
  type.multinomial = "grouped",
  alpha  = 0.5,
  nfolds = 5,
  standardize = TRUE
)

cat(sprintf("lambda.min = %.4f | lambda.1se = %.4f\n",
            fit_test$lambda.min, fit_test$lambda.1se))

# Coefficients au lambda optimal
coefs_min <- coef(fit_test, s = "lambda.min")
cat("\nNombre de coefficients non-nuls par classe :\n")
for (cl in names(coefs_min)) {
  cat(sprintf("  %s : %d / %d (intercept inclus)\n",
              cl, sum(abs(coefs_min[[cl]]) > 1e-8), nrow(coefs_min[[cl]])))
}

# Prédiction test
pred_test <- predict(fit_test, newx = Z_test, s = "lambda.min", type = "class")[, 1]
cm_test <- caret::confusionMatrix(
  factor(pred_test, levels = LABEL_ORDER),
  factor(y_test,    levels = LABEL_ORDER)
)
cat("\n=== Test (sanity check) ===\n")
print(cm_test$table)
cat(sprintf("\nAccuracy : %.3f | Balanced accuracy : %.3f\n",
            cm_test$overall["Accuracy"],
            mean(cm_test$byClass[, "Balanced Accuracy"])))


Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”


lambda.min = 0.1416 | lambda.1se = 0.2475

Nombre de coefficients non-nuls par classe :
  cort : 13 / 16 (intercept inclus)
  dipg : 13 / 16 (intercept inclus)
  midl : 13 / 16 (intercept inclus)

=== Test (sanity check) ===
          Reference
Prediction cort dipg midl
      cort    5    1    1
      dipg    0    5    2
      midl    0    0    0

Accuracy : 0.714 | Balanced accuracy : 0.727


## 6. Cross-validation 7-fold × 3 (protocole identique à NB09/NB10/NB11/NB12)

Pour chaque fold externe :
1. Refit la Sparse PCA sur le train fold uniquement (anti-fuite)
2. Projeter le val fold sur les loadings appris
3. Fit `cv.glmnet` multinomial elastic net (λ choisi par CV interne 5-fold)
4. Prédire le val fold, calculer balanced accuracy

Grid sur α (Elastic Net mixing) : `{0, 0.5, 1}` = ridge, elastic net, lasso.

⚠️ Plus rapide que NB12 parce qu'on n'a pas la double CV de multiview. Compter 5-15 min.

In [6]:
ALPHA_GRID <- c(0, 0.5, 1.0)
cv_results <- data.frame()

set.seed(SEED)
outer_folds <- caret::createMultiFolds(y_train, k = 7, times = 3)

for (alpha_val in ALPHA_GRID) {
  cat(sprintf("\n=== alpha = %.1f ===\n", alpha_val))
  fold_scores <- numeric(length(outer_folds))
  t0_alpha <- Sys.time()

  for (i in seq_along(outer_folds)) {
    tr_idx <- outer_folds[[i]]
    va_idx <- setdiff(seq_along(y_train), tr_idx)

    # Refit Sparse PCA sur train fold uniquement
    spca_ge_f  <- fit_spca(X_ge_train[tr_idx, ],  N_COMP_GE,  KEEP_GE)
    spca_cgh_f <- fit_spca(X_cgh_train[tr_idx, ], N_COMP_CGH, KEEP_CGH)

    Z_tr <- cbind(spca_ge_f$variates,  spca_cgh_f$variates)
    Z_va <- cbind(project_spca(X_ge_train[va_idx, ],  spca_ge_f),
                  project_spca(X_cgh_train[va_idx, ], spca_cgh_f))
    colnames(Z_tr) <- colnames(Z_va) <-
      c(paste0("GE_PC",  seq_len(ncol(spca_ge_f$variates))),
        paste0("CGH_PC", seq_len(ncol(spca_cgh_f$variates))))

    fit_fold <- tryCatch(
      cv.glmnet(x = Z_tr, y = y_train[tr_idx],
                family = "multinomial", type.multinomial = "grouped",
                alpha = alpha_val, nfolds = 5,
                standardize = TRUE),
      error = function(e) { message("  fold ", i, " failed: ", conditionMessage(e)); NULL }
    )
    if (is.null(fit_fold)) { fold_scores[i] <- NA_real_; next }

    pred_va <- predict(fit_fold, newx = Z_va, s = "lambda.min", type = "class")[, 1]
    cm <- caret::confusionMatrix(
      factor(pred_va, levels = LABEL_ORDER),
      factor(y_train[va_idx], levels = LABEL_ORDER)
    )
    fold_scores[i] <- mean(cm$byClass[, "Balanced Accuracy"], na.rm = TRUE)
  }

  elapsed <- difftime(Sys.time(), t0_alpha, units = "mins")
  cat(sprintf("  Bal_acc moy = %.3f ± %.3f  (%.1f min)\n",
              mean(fold_scores, na.rm = TRUE),
              sd(fold_scores, na.rm = TRUE),
              as.numeric(elapsed)))

  cv_results <- rbind(cv_results, data.frame(
    alpha = alpha_val,
    mean_bal_acc = mean(fold_scores, na.rm = TRUE),
    sd_bal_acc   = sd(fold_scores, na.rm = TRUE),
    n_folds      = sum(!is.na(fold_scores))
  ))
}

cat("\n========== RÉSULTATS CV ==========\n")
print(cv_results)

best_idx   <- which.max(cv_results$mean_bal_acc)
best_alpha <- cv_results$alpha[best_idx]
cat(sprintf("\n>>> Meilleur alpha : %.1f  (CV bal_acc = %.3f ± %.3f)\n",
            best_alpha,
            cv_results$mean_bal_acc[best_idx],
            cv_results$sd_bal_acc[best_idx]))



=== alpha = 0.0 ===


Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has f

  Bal_acc moy = 0.636 ± 0.111  (0.6 min)

=== alpha = 0.5 ===


Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message:
“from glmnet C++ code (error code -78); Convergence for 78th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class

  Bal_acc moy = 0.651 ± 0.111  (0.6 min)

=== alpha = 1.0 ===


Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message:
“from glmnet C++ code (error code -50); Convergence for 50th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class

  Bal_acc moy = 0.593 ± 0.151  (0.6 min)

========== RÉSULTATS CV ==========
  alpha mean_bal_acc sd_bal_acc n_folds
1   0.0    0.6361111  0.1114301      21
2   0.5    0.6506614  0.1109857      21
3   1.0    0.5932540  0.1510549      21

>>> Meilleur alpha : 0.5  (CV bal_acc = 0.651 ± 0.111)


## 7. Refit final au meilleur α + évaluation test

In [7]:
set.seed(SEED)

# Refit Sparse PCA sur tout le train
spca_ge_F  <- fit_spca(X_ge_train,  N_COMP_GE,  KEEP_GE)
spca_cgh_F <- fit_spca(X_cgh_train, N_COMP_CGH, KEEP_CGH)
Z_train_F <- cbind(spca_ge_F$variates,  spca_cgh_F$variates)
Z_test_F  <- cbind(project_spca(X_ge_test,  spca_ge_F),
                   project_spca(X_cgh_test, spca_cgh_F))
colnames(Z_train_F) <- colnames(Z_test_F) <-
  c(paste0("GE_PC",  seq_len(ncol(spca_ge_F$variates))),
    paste0("CGH_PC", seq_len(ncol(spca_cgh_F$variates))))

# Fit multinomial elastic net au meilleur alpha
fit_final <- cv.glmnet(
  x = Z_train_F, y = y_train,
  family = "multinomial", type.multinomial = "grouped",
  alpha = best_alpha, nfolds = 5,
  standardize = TRUE
)

cat(sprintf("Refit final : alpha=%.1f, lambda.min=%.4f\n",
            best_alpha, fit_final$lambda.min))

# Coefficients sélectionnés
coefs_final <- coef(fit_final, s = "lambda.min")
cat("\nCoefficients non-nuls par classe :\n")
for (cl in names(coefs_final)) {
  active <- which(abs(coefs_final[[cl]]) > 1e-8)
  active <- active[active > 1]  # retirer l'intercept
  active_names <- rownames(coefs_final[[cl]])[active]
  cat(sprintf("  %s : %d composantes actives - %s\n",
              cl, length(active_names), paste(active_names, collapse = ", ")))
}

# Prédiction test
pred_test_F  <- predict(fit_final, newx = Z_test_F, s = "lambda.min", type = "class")[, 1]
probs_test_F <- predict(fit_final, newx = Z_test_F, s = "lambda.min", type = "response")[, , 1]

cat("\nProbabilités prédites (test set) :\n")
print(round(probs_test_F, 3))

cm_final <- caret::confusionMatrix(
  factor(pred_test_F, levels = LABEL_ORDER),
  factor(y_test,      levels = LABEL_ORDER)
)
cat("\n=== Matrice de confusion test (Sparse PCA + LogReg multinomiale) ===\n")
print(cm_final$table)
cat(sprintf("\nAccuracy : %.3f\n", cm_final$overall["Accuracy"]))
cat(sprintf("Balanced accuracy : %.3f\n",
            mean(cm_final$byClass[, "Balanced Accuracy"])))
cat("\nBal_acc par classe :\n")
print(round(cm_final$byClass[, "Balanced Accuracy"], 3))


Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in mixOmics::spca(X_scaled, ncomp = ncomp, keepX = rep(keep_per_comp, :
“data contain missing values which will be set to zero for calculations. Consider using center = TRUE for better performance, or impute missing values using 'impute.nipals' function.”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has fewer than 8  observations; dangerous ground”
Warning message in lognet(x, is.sparse, y, weights, offset, alpha, nobs, nvars, :
“one multinomial or binomial class has f

Refit final : alpha=0.5, lambda.min=0.1416

Coefficients non-nuls par classe :
  cort : 12 composantes actives - GE_PC1, GE_PC2, GE_PC3, GE_PC4, GE_PC7, GE_PC8, GE_PC9, GE_PC10, CGH_PC1, CGH_PC2, CGH_PC4, CGH_PC5
  dipg : 12 composantes actives - GE_PC1, GE_PC2, GE_PC3, GE_PC4, GE_PC7, GE_PC8, GE_PC9, GE_PC10, CGH_PC1, CGH_PC2, CGH_PC4, CGH_PC5
  midl : 12 composantes actives - GE_PC1, GE_PC2, GE_PC3, GE_PC4, GE_PC7, GE_PC8, GE_PC9, GE_PC10, CGH_PC1, CGH_PC2, CGH_PC4, CGH_PC5

Probabilités prédites (test set) :
     cort  dipg  midl
P08 0.668 0.163 0.169
P09 0.529 0.205 0.266
P11 0.075 0.779 0.146
P14 0.394 0.349 0.258
P19 0.154 0.715 0.132
P21 0.320 0.521 0.159
P24 0.185 0.678 0.137
P28 0.692 0.192 0.116
P29 0.133 0.743 0.124
P39 0.574 0.232 0.194
P42 0.500 0.358 0.142
P44 0.065 0.863 0.071
P47 0.608 0.187 0.205
P51 0.186 0.676 0.137

=== Matrice de confusion test (Sparse PCA + LogReg multinomiale) ===
          Reference
Prediction cort dipg midl
      cort    5    1    1
      dipg 

## 8. Comparaison directe avec NB09, NB11, NB12

| Méthode | Pré-traitement | Classifieur | CV bal_acc | Test bal_acc |
|---|---|---|---|---|
| SGCCA + LDA (NB09) | SGCCA | LDA | 0.829 ± 0.143 | 0.924 |
| Coop argmax (NB11) | Cooperative raw | argmax OvR | 0.833 ± 0.129 | 0.924 |
| sPCA + Coop + argmax (NB12) | sPCA per-block + Cooperative | argmax OvR | _à compléter_ | _à compléter_ |
| **sPCA + LogReg (NB13)** | **sPCA per-block** | **LogReg multinomial EN** | **(ce notebook)** | **(ce notebook)** |

In [8]:
nb09_cv      <- "0.829 ± 0.143";  nb09_test <- 0.924
nb11_cv      <- "0.833 ± 0.129";  nb11_test <- 0.924
# nb12_cv/test à compléter manuellement quand NB12 terminé

nb13_cv_mean <- cv_results$mean_bal_acc[best_idx]
nb13_cv_sd   <- cv_results$sd_bal_acc[best_idx]
nb13_cv      <- sprintf("%.3f ± %.3f", nb13_cv_mean, nb13_cv_sd)
nb13_test    <- mean(cm_final$byClass[, "Balanced Accuracy"])

comparison <- data.frame(
  Notebook = c("NB09 SGCCA+LDA", "NB11 Coop argmax",
               sprintf("NB13 sPCA+LogReg (α=%.1f)", best_alpha)),
  CV_bal_acc   = c(nb09_cv, nb11_cv, nb13_cv),
  Test_bal_acc = c(sprintf("%.3f", nb09_test),
                   sprintf("%.3f", nb11_test),
                   sprintf("%.3f", nb13_test))
)
print(comparison, row.names = FALSE)

cat("\n========== DIAGNOSTIC ==========\n")
delta_vs_nb11 <- nb13_cv_mean - 0.833
cat(sprintf("Δ NB13 - NB11 (Coop raw) : %+.3f\n", delta_vs_nb11))
if (abs(delta_vs_nb11) < 0.02) {
  cat("→ NB13 ≈ NB11 : Sparse PCA + LogReg suffit, cooperative n'apporte rien.\n")
} else if (delta_vs_nb11 > 0) {
  cat("→ NB13 > NB11 : Sparse PCA + LogReg bat même cooperative raw.\n")
} else {
  cat("→ NB13 < NB11 : cooperative apporte quelque chose sur features brutes.\n")
}


                 Notebook    CV_bal_acc Test_bal_acc
           NB09 SGCCA+LDA 0.829 ± 0.143        0.924
         NB11 Coop argmax 0.833 ± 0.129        0.924
 NB13 sPCA+LogReg (α=0.5) 0.651 ± 0.111        0.727

========== DIAGNOSTIC ==========
Δ NB13 - NB11 (Coop raw) : -0.182
→ NB13 < NB11 : cooperative apporte quelque chose sur features brutes.


## 9. Lecture des résultats

Quatre scénarios possibles à l'issue de NB13, par ordre d'intérêt :

### Scénario A — NB13 ≈ NB12 ≈ NB11 (tous ≈ 0.83)

**Le scénario le plus probable et le plus propre.** Tous les pipelines multi-blocs supervisés convergent vers la même performance. Cela montre que :
- L'information utile est dans la sélection sparse des variables
- Le détail méthodologique (cooperative vs LogReg, raw vs Sparse PCA) ne change rien
- La performance est plafonnée par la taille d'échantillon (n=39)

Conclusion pour le rapport : **simplifier le pipeline final**. Présenter NB13 (Sparse PCA + LogReg) comme la **version la plus parcimonieuse** qui atteint le score optimal, et NB09 (SGCCA + LDA) comme la version de référence du paper Tenenhaus.

### Scénario B — NB13 > NB12 ou NB11

**Cooperative learning est délétère.** Sa machinerie additionnelle (agrément cross-blocs) ne sert à rien et peut même dégrader si les hyperparamètres ne sont pas optimaux. Sparse PCA + LogReg suffit.

### Scénario C — NB13 < NB12

**Cooperative apporte quelque chose**, même quand on a déjà Sparse PCA. C'est intéressant : l'agrément cooperative améliore au-delà de la double-sélection sparse.

### Scénario D — NB13 << NB12 (gros écart)

Très improbable. Indiquerait que l'apport de cooperative est massif, ce qui contredirait les résultats de NB07/NB10/NB11.

### Pour ton document de synthèse

Le scénario A est attendu et serait **le résultat final propre** : tous les chemins mènent à la même solution. Tu peux alors écrire :

> *"Quatre pipelines méthodologiquement distincts (SGCCA + LDA, Cooperative + LDA, Cooperative argmax, Sparse PCA + LogReg multinomiale) convergent vers la même balanced accuracy (~0.83 en CV, ~0.92 en test). Cette robustesse aux choix méthodologiques suggère que la performance est limitée par la taille d'échantillon (n=39), pas par les détails algorithmiques. Les méthodes multi-blocs supervisées modernes (cooperative, SGCCA) n'apportent pas de gain mesurable sur ce dataset par rapport à une approche simple Sparse PCA + régression logistique."*

C'est un résultat **plus instructif** pour ton tuteur qu'un score isolé.
